# 🌐 Large-Scale English-Amharic Parallel Corpus Acquisition

A systematic data acquisition pipeline designed to build a high-fidelity parallel corpus of over **17 Million sentence pairs** for training an English $\leftrightarrow$ Amharic Neural Machine Translation (NMT) Transformer.

### 📚 Corpus Inventory & Target Repositories

| Source ID | Repository / Origin | Collected Pool | Alignment & Mining Methodology |
|:---|:---|:---|:---|
| **`nllb`** | Meta AI NLLB Bitext (`amh_Ethi-eng_Latn`) | **16,137,053** | **LASER-3 Semantic Vector Mining**: Streamed directly from official GCS bucket with margin-based cosine similarity. |
| **`mt560`** | `michsethowusu/english-amharic_sentence-pairs_mt560` | **669,145** | Curated OPUS MT560 benchmark slice for extended lexical breadth. |
| **`opus_ccaligned`** | OPUS CCAligned (`am-en`) | **346,511** | Domain & web terminology extracted from Common Crawl. |
| **`opus_tanzil`** | OPUS Tanzil (`am-en`) | **93,526** | Human-translated classical and literary parallel texts with strict sentence alignment. |
| **`opus100`** | `Helsinki-NLP/opus-100` (`am-en`) | **93,027** | Curated multi-domain parallel corpus across news, subtitles, and documentation. |
| **`flores200`** | `rasyosef/flores_english_amharic_mt` | **6,027** | Gold-standard professional human translations (strictly reserved for zero-contamination evaluation). |

---

In [ ]:
# Install required dependencies
!pip install -q datasets pandas pyarrow requests tqdm psutil

In [ ]:
import gc
import gzip
import io
import json
import re
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from datasets import load_dataset

# Environment configuration
SEED = 42
np.random.seed(SEED)

In [ ]:
# Mount Google Drive
base = Path("/content/drive/MyDrive/Internship/MT/Data")

DRIVE_STORAGE_DIR = Path("/content/drive/MyDrive/Internship/MT/Data")

## 📦 Ingestion Engine Architecture
We implement modular loaders with error recovery, streaming capabilities, and robust column normalization. Each loader standardizes output to:
- `english`: Raw English text string
- `amharic`: Raw Amharic text string
- `source`: Dataset identifier tag

In [ ]:
# Regex patterns for script identification
ETHIOPIC_REGEX = re.compile(r"[\u1200-\u137F\u1380-\u139F\u2D80-\u2DDF]")
LATIN_REGEX = re.compile(r"[a-zA-Z]")


def extract_pair(row: dict) -> tuple[str | None, str | None]:
    """
    Linguistically grounded extractor for English and Amharic sentence pairs.
    Uses schema key lookup with automatic Ge'ez script / Latin script fallback,
    guaranteeing successful extraction regardless of column naming conventions.
    """
    en = (
        row.get("english")
        or row.get("English")
        or row.get("eng")
        or row.get("eng_Latn")
        or row.get("sentence_eng_Latn")
    )
    am = (
        row.get("amharic")
        or row.get("Amharic")
        or row.get("amh")
        or row.get("amh_Ethi")
        or row.get("sentence_amh_Ethi")
    )

    # Check nested translation dictionary if present (e.g. OPUS)
    if not en and not am and "translation" in row and isinstance(row["translation"], dict):
        trans = row["translation"]
        en = trans.get("en") or trans.get("eng") or trans.get("eng_Latn")
        am = trans.get("am") or trans.get("amh") or trans.get("amh_Ethi")

    # Fallback 1: Dynamic key scan
    if not en or not am:
        for k, v in row.items():
            if not isinstance(v, str):
                continue
            k_low = k.lower()
            if not en and ("eng" in k_low or k_low.startswith("en") or "en_" in k_low):
                en = v
            elif not am and ("amh" in k_low or k_low.startswith("am") or "am_" in k_low):
                am = v

    # Fallback 2: Script-based content detection (Amharic has Fidel, English has Latin)
    if not en or not am:
        for val in row.values():
            if isinstance(val, str) and val.strip():
                if not am and ETHIOPIC_REGEX.search(val):
                    am = val
                elif not en and LATIN_REGEX.search(val) and not ETHIOPIC_REGEX.search(val):
                    en = val

    if en and am:
        en_str, am_str = str(en).strip(), str(am).strip()
        if en_str and am_str:
            return en_str, am_str
    return None, None


def load_nllb_corpus(max_samples: int | None = None) -> pd.DataFrame:
    """
    Loads Meta's NLLB LASER-mined parallel bitext directly from the official GCS archive:
    https://storage.googleapis.com/allennlp-data-bucket/nllb/amh_Ethi-eng_Latn.gz
    This bypasses Hugging Face datasets 3.0 script deprecation and streams directly.
    """
    t0 = time.time()
    url = "https://storage.googleapis.com/allennlp-data-bucket/nllb/amh_Ethi-eng_Latn.gz"
    print(f"\n[1/6] Streaming Meta NLLB Bitext directly from archive ({url})...")
    try:
        response = requests.get(url, stream=True, timeout=60)
        if response.status_code != 200:
            print(f"   ⚠️ Storage HTTP status: {response.status_code}")
            return pd.DataFrame(columns=["english", "amharic", "source"])

        pairs = []
        with gzip.GzipFile(fileobj=response.raw) as gz:
            for line in gz:
                decoded = line.decode("utf-8", errors="replace").strip()
                if not decoded:
                    continue
                parts = decoded.split("\t")
                if len(parts) >= 2:
                    # In NLLB amh_Ethi-eng_Latn: col 0 is Amharic, col 1 is English
                    am_text, en_text = parts[0].strip(), parts[1].strip()
                    # Ensure orientation matches script
                    if not ETHIOPIC_REGEX.search(am_text) and ETHIOPIC_REGEX.search(en_text):
                        am_text, en_text = en_text, am_text

                    if am_text and en_text:
                        pairs.append((en_text, am_text))

                if max_samples and len(pairs) >= max_samples:
                    break

        df = pd.DataFrame(pairs, columns=["english", "amharic"])
        df["source"] = "nllb"
        elapsed = time.time() - t0
        print(f"    Loaded {len(df):,} pairs from Meta NLLB in {elapsed:.1f}s ({len(df)/max(elapsed, 0.1):,.0f} pairs/sec)")
        return df
    except Exception as e:
        print(f"   ⚠️ Direct NLLB storage fetch note: {e}")
        return pd.DataFrame(columns=["english", "amharic", "source"])


def load_opus_zip_archive(name: str, url: str, max_pairs: int | None = None) -> pd.DataFrame:
    """Downloads and extracts parallel text directly from OPUS CSC storage archives."""
    t0 = time.time()
    print(f"\n[2/6]  Downloading OPUS archive '{name}' from {url}...")
    try:
        response = requests.get(url, stream=True, timeout=60)
        if response.status_code != 200:
            print(f"   ⚠️ HTTP status {response.status_code} received from {url}")
            return pd.DataFrame(columns=["english", "amharic", "source"])

        z = zipfile.ZipFile(io.BytesIO(response.content))
        files = z.namelist()

        en_candidates = [f for f in files if f.endswith(".en") or ".en." in f]
        am_candidates = [f for f in files if f.endswith(".am") or ".am." in f]

        if not en_candidates or not am_candidates:
            print(f"   ⚠️ Could not locate aligned .en and .am files in archive: {files}")
            return pd.DataFrame(columns=["english", "amharic", "source"])

        with z.open(en_candidates[0]) as f_en, z.open(am_candidates[0]) as f_am:
            en_lines = io.TextIOWrapper(f_en, encoding="utf-8", errors="replace").readlines()
            am_lines = io.TextIOWrapper(f_am, encoding="utf-8", errors="replace").readlines()

        total_lines = min(len(en_lines), len(am_lines))
        pairs = []
        limit = max_pairs if max_pairs else total_lines

        for i in range(min(total_lines, limit)):
            e = en_lines[i].strip()
            a = am_lines[i].strip()
            if e and a:
                pairs.append((e, a))

        df = pd.DataFrame(pairs, columns=["english", "amharic"])
        df["source"] = f"opus_{name}"
        elapsed = time.time() - t0
        print(f"    Loaded {len(df):,} pairs from OPUS {name} in {elapsed:.1f}s")
        return df
    except Exception as e:
        print(f"   ⚠️ OPUS archive '{name}' download error: {e}")
        return pd.DataFrame(columns=["english", "amharic", "source"])


def load_opus100_corpus() -> pd.DataFrame:
    """Loads Helsinki-NLP/opus-100 (am-en, 93k pairs across train/validation/test)."""
    t0 = time.time()
    print("\n[3/6]  Fetching Helsinki-NLP/opus-100 (am-en)...")
    try:
        ds = load_dataset("Helsinki-NLP/opus-100", "am-en")
        pairs = []
        for split in ["train", "validation", "test"]:
            if split in ds:
                for row in ds[split]:
                    en, am = extract_pair(row)
                    if en and am:
                        pairs.append((en, am))

        df = pd.DataFrame(pairs, columns=["english", "amharic"])
        df["source"] = "opus100"
        elapsed = time.time() - t0
        print(f"    Loaded {len(df):,} pairs from OPUS-100 in {elapsed:.1f}s")
        return df
    except Exception as e:
        print(f"   ⚠️ Failed to load OPUS-100: {e}")
        return pd.DataFrame(columns=["english", "amharic", "source"])


def load_mt560_corpus(max_samples: int | None = None) -> pd.DataFrame:
    """Loads the curated MT560 English-Amharic dataset (669,145 pairs)."""
    t0 = time.time()
    print("\n[4/6]  Fetching MT560 Curated Corpus (michsethowusu/english-amharic_sentence-pairs_mt560)...")
    try:
        ds = load_dataset("michsethowusu/english-amharic_sentence-pairs_mt560", split="train")
        if max_samples and max_samples < len(ds):
            ds = ds.select(range(max_samples))

        pairs = []
        for row in ds:
            en, am = extract_pair(row)
            if en and am:
                pairs.append((en, am))

        df = pd.DataFrame(pairs, columns=["english", "amharic"])
        df["source"] = "mt560"
        elapsed = time.time() - t0
        print(f"    Loaded {len(df):,} pairs from MT560 in {elapsed:.1f}s ({len(df)/max(elapsed, 0.1):,.0f} pairs/sec)")
        return df
    except Exception as e:
        print(f"   ⚠️ Failed to load MT560: {e}")
        return pd.DataFrame(columns=["english", "amharic", "source"])


def load_drive_custom_corpus(
    am_path: str | Path | None = None,
    en_path: str | Path | None = None
) -> pd.DataFrame:
    """Loads verified human-curated sentence pairs from Google Drive or local path."""
    print("\n[5/6]  Ingesting verified custom Drive corpus...")
    raw_am = Path("/content/drive/MyDrive/Internship/MT/Data/am_data.txt")
    raw_en = Path("/content/drive/MyDrive/Internship/MT/Data/eng_data.txt")

    target_am = Path(raw_am)
    target_en = Path(raw_en)

    if not (target_am.is_file() and target_en.is_file()):
        print(f"   ℹ️ Custom drive files not found at: {target_am}. Skipping custom loader.")
        return pd.DataFrame(columns=["english", "amharic", "source"])

    t0 = time.time()
    try:
        with open(target_am, encoding="utf-8", errors="replace") as f_am:
            am_lines = f_am.readlines()
        with open(target_en, encoding="utf-8", errors="replace") as f_en:
            en_lines = f_en.readlines()

        total = min(len(am_lines), len(en_lines))
        pairs = []
        for i in range(total):
            a, e = am_lines[i].strip(), en_lines[i].strip()
            if a and e:
                pairs.append((e, a))

        df = pd.DataFrame(pairs, columns=["english", "amharic"])
        df["source"] = "drive_custom"
        elapsed = time.time() - t0
        print(f"   ✅ Loaded {len(df):,} verified custom pairs from {target_am.parent.name} in {elapsed:.1f}s")
        return df
    except Exception as e:
        print(f"   ⚠️ Error reading custom files: {e}")
        return pd.DataFrame(columns=["english", "amharic", "source"])


def load_flores_benchmark() -> pd.DataFrame:
    """
    Loads gold-standard human-translated evaluation benchmark (FLORES-200).
    Kept strictly isolated from training corpora to prevent data contamination.
    """
    t0 = time.time()
    print("\n[6/6] 📥 Fetching FLORES-200 Gold Evaluation Benchmark...")
    try:
        ds = load_dataset("rasyosef/flores_english_amharic_mt")
        pairs = []
        for split in ds:
            for row in ds[split]:
                en, am = extract_pair(row)
                if not en or not am:
                    for v in row.values():
                        if isinstance(v, str) and v.strip():
                            if not am and ETHIOPIC_REGEX.search(v):
                                am = v.strip()
                            elif not en and LATIN_REGEX.search(v) and not ETHIOPIC_REGEX.search(v):
                                en = v.strip()

                if en and am:
                    pairs.append({"english": en, "amharic": am, "source": "flores200", "split": split})

        df = pd.DataFrame(pairs)
        elapsed = time.time() - t0
        print(f"   ✅ Loaded {len(df):,} gold evaluation pairs from FLORES-200 in {elapsed:.1f}s")
        return df
    except Exception as e:
        print(f"   ⚠️ FLORES benchmark note: {e}")
        return pd.DataFrame(columns=["english", "amharic", "source", "split"])

## 🚀 Parallel Corpus Collection Execution
We now execute the collection routines with semantically aligned sources. The pipeline emphasizes high-precision semantic pairs (Meta NLLB, OPUS collections, and custom data).

In [ ]:
# ==============================================================================
# COLLECTION SOURCES CONFIGURATION
# ==============================================================================
INCLUDE_NLLB        = True        # Meta NLLB LASER-mined bitext (16.1M pairs pool)
INCLUDE_CCALIGNED   = True        # OPUS CCAligned (~346k pairs)
INCLUDE_TANZIL      = True        # OPUS Tanzil (~94k pairs)
INCLUDE_OPUS100     = True        # OPUS-100 (~93k pairs)
INCLUDE_MT560       = True        # Curated MT560 slice (669k pairs)
CHECK_CUSTOM_DRIVE  = True        # User verified Drive dataset (~228k pairs)

CUSTOM_AM_PATH = Path("/content/drive/MyDrive/Internship/MT/Data/am_data.txt")
CUSTOM_EN_PATH = Path("/content/drive/MyDrive/Internship/MT/Data/eng_data.txt")
# ==============================================================================

pipeline_start = time.time()
collected_dfs = []

# 1. Meta NLLB LASER-mined Bitext (Direct GCS Archive Stream)
if INCLUDE_NLLB:
    df_nllb = load_nllb_corpus()
    if len(df_nllb) > 0:
        collected_dfs.append(df_nllb)

# 2. OPUS CCAligned
if INCLUDE_CCALIGNED:
    df_ccaligned = load_opus_zip_archive(
        name="ccaligned",
        url="https://object.pouta.csc.fi/OPUS-CCAligned/v1/moses/am-en.txt.zip"
    )
    if len(df_ccaligned) > 0:
        collected_dfs.append(df_ccaligned)

# 3. OPUS Tanzil
if INCLUDE_TANZIL:
    df_tanzil = load_opus_zip_archive(
        name="tanzil",
        url="https://object.pouta.csc.fi/OPUS-Tanzil/v1/moses/am-en.txt.zip"
    )
    if len(df_tanzil) > 0:
        collected_dfs.append(df_tanzil)

# 4. OPUS-100 Multi-Domain
if INCLUDE_OPUS100:
    df_opus100 = load_opus100_corpus()
    if len(df_opus100) > 0:
        collected_dfs.append(df_opus100)

# 5. MT560 Curated Slice
if INCLUDE_MT560:
    df_mt560 = load_mt560_corpus()
    if len(df_mt560) > 0:
        collected_dfs.append(df_mt560)

# 6. Local / Drive Custom Data
if CHECK_CUSTOM_DRIVE:
    df_drive = load_drive_custom_corpus(am_path=CUSTOM_AM_PATH, en_path=CUSTOM_EN_PATH)
    if len(df_drive) > 0:
        collected_dfs.append(df_drive)

# 7. FLORES-200 Evaluation Benchmark (Saved separately for zero contamination)
df_flores = load_flores_benchmark()

# Concatenate all training sources
print("\n" + "=" * 70)
print("🔄 CONSOLIDATING COLLECTED PARALLEL CORPORA...")
print("=" * 70)

if collected_dfs:
    df_raw = pd.concat(collected_dfs, ignore_index=True)
    df_raw["source"] = df_raw["source"].astype("category")
else:
    df_raw = pd.DataFrame(columns=["english", "amharic", "source"])

total_seconds = time.time() - pipeline_start
mem_mb = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"\n🎉 ACQUISITION COMPLETE in {total_seconds/60:.2f} minutes!")
print(f"• Total Parallel Sentence Pairs Collected : {len(df_raw):,}")
print(f"• In-Memory DataFrame Size                : {mem_mb:.2f} MB")
print("\n📊 Breakdown by Source:")
print("-" * 45)
for src, count in df_raw["source"].value_counts().items():
    pct = (count / len(df_raw)) * 100
    print(f"  {src:<22}: {count:>10,} pairs ({pct:>5.1f}%)")
print("-" * 45)

## 🔍 Parallel Integrity & Script Sanity Checks
Before writing to permanent storage, we run integrity checks across all collected sentence pairs:
1. **Parallel Completeness**: Zero null, empty, or whitespace-only records.
2. **Identity Filtering**: Eliminate cases where the Amharic and English fields are identical.
3. **Ethiopic Script Compliance**: Verify the presence of Ge'ez / Fidel characters (`\u1200`–`\u137F`).
4. **Latin Script Compliance**: Verify English text contains standard Latin characters.

In [ ]:
# Vectorized Script & Content Verification
print("=" * 70)
print("🧪 PARALLEL CORPUS INTEGRITY VERIFICATION")
print("=" * 70)

total_pairs = len(df_raw)
if total_pairs > 0:
    # 1. Null and empty string checks
    en_valid = df_raw["english"].notna() & (df_raw["english"].astype(str).str.strip() != "")
    am_valid = df_raw["amharic"].notna() & (df_raw["amharic"].astype(str).str.strip() != "")
    valid_mask = en_valid & am_valid

    # 2. Direct equality check (untranslated / identical pairs)
    diff_mask = df_raw["english"].astype(str).str.strip() != df_raw["amharic"].astype(str).str.strip()

    sample_size = min(200_000, total_pairs)
    sample_indices = np.random.choice(total_pairs, size=sample_size, replace=False)
    sample_df = df_raw.iloc[sample_indices]

    am_has_fidel = sample_df["amharic"].apply(lambda x: bool(ETHIOPIC_REGEX.search(str(x))))
    en_has_latin = sample_df["english"].apply(lambda x: bool(LATIN_REGEX.search(str(x))))

    print(f"• Total Candidates Evaluated         : {total_pairs:,}")
    print(f"• Both Sides Non-Empty               : {valid_mask.sum():,} ({(valid_mask.sum()/total_pairs)*100:.2f}%)")
    print(f"• Distinct Source/Target Pairs       : {diff_mask.sum():,} ({(diff_mask.sum()/total_pairs)*100:.2f}%)")
    print(f"• Sample Script Compliance (n={sample_size:,}):")
    print(f"  - Amharic contains Fidel characters : {am_has_fidel.mean() * 100:.2f}%")
    print(f"  - English contains Latin characters : {en_has_latin.mean() * 100:.2f}%")

    # Word-length statistics
    en_words = sample_df["english"].str.split().str.len()
    am_words = sample_df["amharic"].str.split().str.len()
    print("\n📏 Length Profile (Sample Statistics):")
    print(f"  - English Words : mean={en_words.mean():.1f}, median={en_words.median():.1f}, p95={en_words.quantile(0.95):.0f}")
    print(f"  - Amharic Words : mean={am_words.mean():.1f}, median={am_words.median():.1f}, p95={am_words.quantile(0.95):.0f}")
print("=" * 70)

In [ ]:
# Side-by-Side Sample Inspection Across Every Source
print("=" * 70)
print(" SIDE-BY-SIDE PARALLEL SAMPLES BY SOURCE")
print("=" * 70)

for src in df_raw["source"].unique():
    subset = df_raw[df_raw["source"] == src]
    sample_n = min(3, len(subset))
    samples = subset.sample(sample_n, random_state=SEED)

    print(f"\n📁 Source: [{src}] (Total pool: {len(subset):,} pairs)")
    print("-" * 70)
    for idx, (_, row) in enumerate(samples.iterrows(), 1):
        en_preview = row["english"][:110] + ("..." if len(row["english"]) > 110 else "")
        am_preview = row["amharic"][:110] + ("..." if len(row["amharic"]) > 110 else "")
        print(f"  [{idx}] EN: {en_preview}")
        print(f"      AM: {am_preview}")

Direct-to-Drive Chunked Parquet Serialization

In [ ]:
print("=" * 70)
print(f" DIRECT-TO-DRIVE SERIALIZATION (Target: {DRIVE_STORAGE_DIR})")
print("=" * 70)

parquet_path = DRIVE_STORAGE_DIR / "raw_parallel_corpus.parquet"
sample_tsv_path = DRIVE_STORAGE_DIR / "raw_parallel_corpus_sample_100k.tsv"
flores_path = DRIVE_STORAGE_DIR / "flores200_benchmark.parquet"
manifest_path = DRIVE_STORAGE_DIR / "collection_manifest.json"

# 1. Chunked ParquetWriter directly to Google Drive
t0 = time.time()
print(f"• Serializing {len(df_raw):,} pairs directly to Drive in streaming batches...")

schema = pa.schema([
    ("english", pa.string()),
    ("amharic", pa.string()),
    ("source", pa.string())
])

CHUNK_SIZE = 500_000
total_rows = len(df_raw)

with pq.ParquetWriter(parquet_path, schema, compression="snappy") as writer:
    for start_idx in range(0, total_rows, CHUNK_SIZE):
        end_idx = min(start_idx + CHUNK_SIZE, total_rows)
        chunk_df = df_raw.iloc[start_idx:end_idx]
        table = pa.Table.from_pandas(chunk_df, schema=schema)
        writer.write_table(table)
        del table
        del chunk_df
        gc.collect()
        pct = (end_idx / total_rows) * 100
        print(f"   → Progress: rows {start_idx:,} to {end_idx:,} / {total_rows:,} ({pct:.1f}%)")

parquet_size_mb = parquet_path.stat().st_size / (1024 ** 2)
print(f"\n✅ Raw Parallel Corpus Saved: {parquet_path.name} ({parquet_size_mb:.2f} MB in {time.time()-t0:.1f}s)")

# 2. Export 100k Sample TSV directly to Drive for inspection
print("• Exporting 100k inspection TSV to Drive...")
df_raw.head(100_000).to_csv(sample_tsv_path, sep="\t", index=False)
tsv_size_mb = sample_tsv_path.stat().st_size / (1024 ** 2)
print(f"✅ Inspection Sample Saved: {sample_tsv_path.name} ({tsv_size_mb:.2f} MB)")

# 3. Export FLORES-200 benchmark directly to Drive
if len(df_flores) > 0:
    df_flores.to_parquet(flores_path, engine="pyarrow", compression="snappy", index=False)
    print(f"✅ FLORES-200 Benchmark Saved: {flores_path.name} ({len(df_flores):,} gold pairs)")

# 4. Generate Metadata Manifest directly in Drive
manifest = {
    "task": "English-Amharic Parallel Corpus Acquisition",
    "creation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "total_parallel_pairs": int(len(df_raw)),
    "storage": {
        "parquet_file": str(parquet_path.name),
        "parquet_size_mb": round(parquet_size_mb, 2),
        "sample_tsv": str(sample_tsv_path.name),
        "drive_directory": str(DRIVE_STORAGE_DIR),
    },
    "sources": df_raw["source"].value_counts().to_dict(),
    "benchmark": {
        "flores200_pairs": int(len(df_flores)) if len(df_flores) > 0 else 0
    }
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"✅ Metadata Manifest Saved: {manifest_path.name}")
print("\n📄 Manifest Preview:")
print(json.dumps(manifest, indent=2))
print("=" * 70)